# 05. Corpus ampliado

Ampliamos el corpus con las normas que el banco cita y no teníamos, y volvemos a
recorrer el pipeline completo: ingesta, índice, recuperación, generación y evaluación.
El 03 y el 04 quedan como están.

Separamos tres corridas para no confundir causas:

| Corrida | Corpus | Generación | Decoder | Qué aísla |
|---|---|---|---|---|
| A | ampliado | igual al 04 | Salamandra-7B | el aporte del corpus |
| B | ampliado | saneo por cita y fundamento con normas usadas | Salamandra-7B | el aporte de los arreglos de generación |
| C | ampliado | igual a B | Qwen3-8B | el aporte del modelo |

La corrida C está apagada por defecto. El enunciado sugiere Qwen3-8B por nombre, pero
el modelo tiene 8.190.735.360 parámetros y el límite del reto es de 8.000 millones:
hay que confirmarlo con el organizador antes de usarlo en la entrega.

Con 50 preguntas, una o dos cerradas de diferencia son ruido. Lo que compara bien es
la misma variante entre versiones.

## Entorno

El mismo notebook corre en local o en Colab. En Colab hay que subir `ai-week.zip`
a `Mi unidad/AIWEEK` sin descomprimir y elegir L4 o A100.

Para comparar contra el 03 y el 04 hay que copiar `AIWEEK/experimentos` de Drive a
`data/experimentos_sistema`. Si no está, la tabla de evolución omite esas filas.

La llave del juez va en `data/oficial/scripts/.env`, con la línea
`OPENROUTER_API_KEY=...`. Ese archivo no se versiona.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import platform
import shutil
import subprocess
import sys
import zipfile

MODO = "auto"
CARPETA_DRIVE = "/content/drive/MyDrive/AIWEEK"

os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
os.environ["USE_FLAX"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

try:
    import google.colab
    detectado_colab = True
except ImportError:
    detectado_colab = False
if MODO not in {"auto", "local", "colab"}:
    raise ValueError("MODO debe ser auto, local o colab")
EN_COLAB = detectado_colab if MODO == "auto" else MODO == "colab"

def hash_archivo(ruta):
    h = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(8 * 1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path(CARPETA_DRIVE)
    paquete = DRIVE / "ai-week.zip"
    if not paquete.is_file():
        raise FileNotFoundError(f"Subir ai-week.zip a {DRIVE}")
    identificador = hash_archivo(paquete)
    RAIZ = Path("/content/aiweek") / ("v05-" + identificador[:12])
    if not (RAIZ / ".paquete_verificado").is_file():
        RAIZ.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(paquete) as z:
            for nombre in z.namelist():
                if not (RAIZ / nombre).resolve().is_relative_to(RAIZ.resolve()):
                    raise ValueError("Ruta inválida dentro del ZIP")
            z.extractall(RAIZ)
        inventario = json.loads((RAIZ / "paquete.json").read_text(encoding="utf-8"))
        for nombre, esperado in inventario["sha256_archivos"].items():
            if hash_archivo(RAIZ / nombre) != esperado:
                raise ValueError(f"Archivo incompleto: {nombre}")
        (RAIZ / ".paquete_verificado").write_text(identificador)
    if not (RAIZ / "scripts/experimentos/v05.py").is_file():
        raise FileNotFoundError("El ZIP no trae scripts/experimentos/v05.py. Regenerarlo")
    dependencias = RAIZ / "requirements-colab.txt"
    firma = hash_archivo(dependencias) + platform.python_version()
    marca = Path("/content/aiweek_v05_dependencias.txt")
    if not marca.exists() or marca.read_text() != firma:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(dependencias)], check=True)
        marca.write_text(firma)
    for nombre in ("numpy", "pandas", "transformers", "tokenizers"):
        if nombre in sys.modules and getattr(sys.modules[nombre], "__version__", None) != importlib.metadata.version(nombre):
            raise RuntimeError("Reiniciar la sesión de Colab y ejecutar desde el principio")
    RESULTADOS = DRIVE / "experimentos"
    FUENTES_INDICES = [DRIVE / "resultados", RAIZ / "data/index"]
    CACHE_MODELOS = DRIVE / "modelos_cache"
    os.environ["HF_HOME"] = str(RAIZ / "data/cache_huggingface")
else:
    RAIZ = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/experimentos.json").is_file()), None)
    if RAIZ is None:
        raise FileNotFoundError("Abrir el notebook desde el proyecto o notebooks")
    RESULTADOS = RAIZ / "data/experimentos_sistema"
    FUENTES_INDICES = [RAIZ / "data/index", RAIZ / "data/experimentos_oficiales"]
    CACHE_MODELOS = None

for nombre, modulo in list(sys.modules.items()):
    if nombre == "scripts" or nombre.startswith("scripts."):
        archivo = getattr(modulo, "__file__", None)
        rutas = [archivo] if archivo else list(getattr(modulo, "__path__", []))
        if any(not Path(r).resolve().is_relative_to(RAIZ.resolve()) for r in rutas):
            raise RuntimeError("Hay código de otro paquete cargado. Reiniciar la sesión y ejecutar desde el inicio")

os.chdir(RAIZ)
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
RESULTADOS.mkdir(parents=True, exist_ok=True)
print("Entorno:", "Colab" if EN_COLAB else "local")
print("Resultados:", RESULTADOS)

## Configuración

El pipeline de experimentos lee `configs/experimentos.json`, no `configs/indice.json`.
Ese segundo archivo solo lo usa `scripts.indice.construir` desde la línea de comandos.
Hoy los dos declaran la misma ventana y el mismo solapamiento; la celda lo comprueba y
avisa si dejan de coincidir.

| Parámetro | Valor |
|---|---|
| Encoder | BGE-M3, ventana 320, solapamiento 32 |
| Índice | FAISS IndexFlatIP, denso |
| Decoder | `salamandra-7b-instruct`, Q4_K_M |
| Temperatura | 0, semilla 0 |
| Variantes | densa, híbrida (BM25 en la consulta), densa con reranker |
| Presupuesto | 22 s por pregunta |

`descargar`, `reingestar` y `usar_juez` controlan los pasos costosos. Los dejamos
explícitos para poder correr el notebook por tramos.

El lote se ajusta a la GPU. BGE-M3 en fp16 llegó a 1,2 GiB de VRAM con lote 4, así que
en una tarjeta grande conviene subirlo: el índice completo son unas 90.000 ventanas.

In [ ]:
import pandas as pd
import torch
from IPython.display import display

from scripts.entorno.runtime import hardware
from scripts.experimentos import v04, v05

descargar = True
reingestar = True
permitir_construir = True
correr_C = False
usar_juez = False
IDS_PRUEBA = [51, 253, 617, 919, 946, 1073]

experimentos_cfg = json.loads((RAIZ / "configs/experimentos.json").read_text(encoding="utf-8"))
indice_cfg = json.loads((RAIZ / "configs/indice.json").read_text(encoding="utf-8"))
for clave in ("tamano_tokens", "solapamiento_tokens"):
    if experimentos_cfg[clave] != indice_cfg[clave]:
        print(f"AVISO: {clave} difiere entre configs/experimentos.json ({experimentos_cfg[clave]}) "
              f"y configs/indice.json ({indice_cfg[clave]}). Manda el primero.")
variantes = [v["nombre"] for v in experimentos_cfg["variantes"]]
device = "cuda" if torch.cuda.is_available() else "cpu"
memoria = torch.cuda.get_device_properties(0).total_memory if device == "cuda" else 0
batch_size = 32 if memoria >= 14 * 2**30 else 8 if memoria >= 3.5 * 2**30 else 4
# Salamandra Q4_K_M pesa 4,6 GiB. En una GPU de 4 GB la prueba de humo va con qwen3-4b
# solo para ejercitar la tubería; sus cifras no se comparan con las de la entrega.
decoder_prueba = None if memoria >= 6 * 2**30 else "qwen3-4b"
print(hardware() if device == "cuda" else "CPU")
print("Encoder:", experimentos_cfg["encoder"], "| ventana:", experimentos_cfg["tamano_tokens"],
      "| solapamiento:", experimentos_cfg["solapamiento_tokens"], "| lote:", batch_size)
print("Variantes:", variantes)

CARPETA_03 = next(iter(sorted((RESULTADOS / "ejecuciones").glob("*"), reverse=True)), None)
CARPETA_04 = RESULTADOS / "v04" / "salamandra_v04_a100"
for etiqueta, carpeta in [("03", CARPETA_03), ("04", CARPETA_04)]:
    estado = "encontrada" if carpeta and Path(carpeta).is_dir() else "no está (copiar desde Drive)"
    print(etiqueta, estado, carpeta)

## 1. Normas que faltan

`scripts.corpus.ampliar` compara el corpus con dos fuentes de objetivos: `seed_targets.json`,
que declara las normas que cita el banco de 992, y el fundamento de la muestra de 50,
que cita algunas que no están en ese archivo. De ahí arma la URL oficial:

- Corte Constitucional: relatoría. Las C y T llevan guion (`c-207-19`), las SU no (`su016-20`).
- Leyes y decretos: Secretaría del Senado. Las normas largas están partidas en varias
  páginas y seguimos la cadena `_prNNN`.
- Corte Suprema y acuerdos: sin patrón. `scripts.corpus.fuentes_csj` los busca en el
  índice del portal; lo que no aparece queda en `data/raw/fuentes_manuales_v05.csv`.

Cada descarga se valida por identificador, también en los PDF. Una página de error que
responde 200 no entra al corpus.

In [ ]:
plan = subprocess.run([sys.executable, "-m", "scripts.corpus.ampliar", "--solo-listar"],
                      capture_output=True, text=True, encoding="utf-8")
print(plan.stdout[:3000] or plan.stderr[-2000:])
manuales = pd.read_csv(RAIZ / "data/raw/fuentes_manuales_v05.csv")
display(manuales[["norma", "items_del_banco", "url", "motivo_manual"]])

In [ ]:
if descargar:
    for modulo in ("scripts.corpus.fuentes_csj", "scripts.corpus.ampliar"):
        proceso = subprocess.run([sys.executable, "-m", modulo],
                                 capture_output=True, text=True, encoding="utf-8")
        print(proceso.stdout[-2500:] or proceso.stderr[-2000:])
    informe = pd.read_csv(RAIZ / "data/raw/ampliacion_v05.csv")
    display(informe["resultado"].value_counts().to_frame("descargas"))
    display(informe[~informe["resultado"].str.startswith("ok")][["norma", "items_del_banco", "resultado"]])

## 2. Ingesta

Guardamos una copia del corpus del 04 en `data/processed/corpus_v04` la primera vez, para
que el 03 y el 04 sigan siendo reproducibles. Después procesamos todo el manifiesto. Si
alguna comprobación falla, no escribimos nada.

In [ ]:
from scripts.corpus.ingesta import ejecutar_ingesta, guardar_resultados

CORPUS = RAIZ / "data/processed/corpus"
COPIA_04 = RAIZ / "data/processed/corpus_v04"
if not COPIA_04.exists():
    shutil.copytree(CORPUS, COPIA_04)
    print("Copia del corpus anterior en", COPIA_04)

if reingestar:
    resultado = ejecutar_ingesta(RAIZ, modo="corpus")
    comprobaciones = pd.DataFrame(resultado["comprobaciones"])
    fallidas = comprobaciones[comprobaciones["resultado"] != "ok"]
    errores = [r for r in resultado["registros"] if r["estado_extraccion"] == "error"]
    print("Comprobaciones:", len(comprobaciones), "| fallidas:", len(fallidas))
    print("Documentos:", resultado["resumen"]["documentos"], "| con error:", len(errores))
    display(fallidas)
    display(pd.DataFrame([{"doc_id": r["doc_id"], "error": r["error"]} for r in errores]))

In [ ]:
if reingestar and fallidas.empty and not errores:
    guardar_resultados(CORPUS, resultado)
    print("Corpus:", resultado["resumen"]["documentos"], "documentos,",
          resultado["resumen"]["unidades"], "unidades,", resultado["resumen"]["fragmentos"], "ventanas")
    display(pd.Series(resultado["resumen"]["unidades_por_tipo"], name="unidades").to_frame())

## 3. Cobertura antes y después

El punto de partida del 04, con la identidad de los códigos corregida, era 48 de 61 citas
de la muestra presentes y 388 de 559 menciones del banco. La cobertura no dice que el
artículo recuperado sea el correcto: solo que la norma está en el corpus.

In [ ]:
from scripts.corpus.cobertura import auditar, auditar_banco, indice_corpus

preguntas, entradas = v04.muestra(RAIZ)
indice = indice_corpus(RAIZ)
cobertura = pd.DataFrame(auditar(RAIZ, preguntas, indice=indice))
print("Muestra de 50, citas de referencia:", cobertura["estado_corpus"].value_counts().to_dict())
display(cobertura[cobertura["estado_corpus"].isin(["norma_ausente", "articulo_ausente"])]
        [["id", "area", "cuerpo", "articulo", "estado_corpus"]])

In [ ]:
banco = pd.DataFrame(auditar_banco(RAIZ, indice=indice))
total, cubierto = banco["items_del_banco"].sum(), banco.loc[banco.en_corpus, "items_del_banco"].sum()
print(f"Banco: {cubierto}/{total} menciones ({100 * cubierto / total:.1f} %) | "
      f"normas {int(banco.en_corpus.sum())}/{len(banco)}")

por_area = banco.explode("areas").groupby("areas").apply(
    lambda g: pd.Series({"normas": len(g), "normas_en_corpus": int(g.en_corpus.sum()),
                         "menciones": g.items_del_banco.sum(),
                         "cubiertas": g.loc[g.en_corpus, "items_del_banco"].sum()}), include_groups=False)
por_area["cobertura"] = (100 * por_area.cubiertas / por_area.menciones).round(1)
display(por_area.sort_values("menciones", ascending=False))
display(banco[~banco.en_corpus].sort_values("items_del_banco", ascending=False)
        .head(15)[["norma", "items_del_banco", "donde_buscar"]])

## 4. Índice y recuperación

El corpus cambió, así que reconstruimos el índice BGE-M3 desde cero y recuperamos otra vez
las 50 preguntas para las tres variantes. El índice es denso; la variante híbrida engancha
BM25 en la consulta y la del reranker reordena veinte candidatos.

In [ ]:
from scripts.experimentos.orquestador import preparar_recuperaciones

CARPETA_RECUP = preparar_recuperaciones(
    RAIZ, RESULTADOS, fuentes_indices=FUENTES_INDICES, variantes=variantes,
    device=device, batch_size=batch_size, permitir_construir=permitir_construir)
print("Recuperación:", CARPETA_RECUP)

ev = v04.evidencia(RAIZ)
recups = {v: {i: ev.reconstruir(p)[0] for i, p in v04.recuperacion_03(CARPETA_RECUP, v).items()}
          for v in variantes}
cobertura_r = pd.DataFrame(auditar(RAIZ, preguntas, recuperaciones=recups, indice=indice))
con_cita = cobertura_r[cobertura_r["estado_corpus"] != "sin_cita_extraible"]
display(pd.concat({v: con_cita[v].value_counts() for v in variantes}, axis=1).fillna(0).astype(int))

## 5. Corrida A: corpus nuevo, generación del 04

Primero seis preguntas de prueba, para ver el formato y el motivo de corte antes de
gastar la tanda completa. Cada respuesta se guarda al terminar; si el código o la
configuración cambian con el mismo nombre de corrida, se detiene en vez de mezclar.

Salamandra Q4_K_M pesa 4,6 GiB. En una GPU de 4 GB la prueba cambia a `qwen3-4b` para
ejercitar la tubería, y las tandas completas de abajo hay que correrlas en Colab: con
descarga parcial a CPU los tiempos por pregunta no dicen nada del presupuesto real.

In [ ]:
prueba = v05.correr(RAIZ, CARPETA_RECUP, RESULTADOS, ["densa"], corrida="A",
                    solo_ids=IDS_PRUEBA, decoder=decoder_prueba)
print("Decoder de la prueba:", decoder_prueba or "salamandra-7b-instruct")
for i in IDS_PRUEBA:
    fila = json.loads((prueba / "densa" / "respuestas" / f"{i}.json").read_text(encoding="utf-8"))
    r = fila["respuesta"]
    print(f"\n{i} {r['formato']} | fin: {fila['registro'].get('finish_reason')} | "
          f"{r.get('latencia_ms', 0) / 1000:.1f} s")
    display({k: v for k, v in r.items() if k != "pasajes_recuperados"})

In [ ]:
CARPETA_A = v05.correr(RAIZ, CARPETA_RECUP, RESULTADOS, variantes, corrida="A",
                       cache_modelos=CACHE_MODELOS)
display(pd.DataFrame({v: json.loads((CARPETA_A / v / "resumen.json").read_text(encoding="utf-8"))
                      for v in variantes}).T)

## 6. Corrida B: saneo por cita y fundamento con normas usadas

Dos cambios sobre la corrida A:

- `saneo="cita"`: cuando una oración cita una norma que no está en los pasajes,
  reemplazamos solo la cita y conservamos el razonamiento. En el 04 borrábamos la oración
  entera y a veces quedaba una frase de relleno.
- `citar_evidencia="usadas"`: el fundamento nombra las normas que el modelo usó más la
  primera recuperada, no todas las recuperadas. Citar menos baja la tasa de citas sin
  respaldo, que pesa el doble en el puntaje.

In [ ]:
CARPETA_B = v05.correr(RAIZ, CARPETA_RECUP, RESULTADOS, variantes, corrida="B",
                       cache_modelos=CACHE_MODELOS)
display(pd.DataFrame({v: json.loads((CARPETA_B / v / "resumen.json").read_text(encoding="utf-8"))
                      for v in variantes}).T)

## 7. Corrida C: decoder Qwen3-8B

Apagada por defecto. El enunciado la sugiere por nombre en su sección 3.1, pero fija el
límite en 8.000 millones de parámetros y el modelo tiene 8.190.735.360; por eso está en
`excluidos` de `configs/modelos.json`. Hay que preguntarle al organizador antes de usarla
en la entrega, y el GGUF no está descargado.

In [ ]:
CARPETA_C = None
if correr_C:
    CARPETA_C = v05.correr(RAIZ, CARPETA_RECUP, RESULTADOS, ["densa"], corrida="C",
                           cache_modelos=CACHE_MODELOS, confirmado_8b=True)
    display(pd.DataFrame([json.loads((CARPETA_C / "densa" / "resumen.json").read_text(encoding="utf-8"))]))

## 8. Segundos por pregunta

El sábado son 992 preguntas en unas seis horas: unos 22 s por pregunta, contando
recuperación y generación. Medimos la corrida completa, no la prueba de seis.

In [ ]:
for etiqueta, carpeta in [("A", CARPETA_A), ("B", CARPETA_B), ("C", CARPETA_C)]:
    if carpeta is None:
        continue
    for variante in variantes if carpeta is not CARPETA_C else ["densa"]:
        if not (Path(carpeta) / variante / "respuestas").is_dir():
            continue
        tabla = v05.tiempos(carpeta, variante, CARPETA_RECUP)
        resumen = v05.resumen_tiempos(tabla)
        resumen.insert(0, "corrida", f"{etiqueta} {variante}")
        display(resumen)

## 9. Evolución 03 → 04 → 05

Sin juez, la comparación cubre 50 de los 80 puntos automáticos. Los 30 del texto libre
solo aparecen si corremos el juez.

In [ ]:
versiones = {"03": CARPETA_03, "04": CARPETA_04, "05A": CARPETA_A, "05B": CARPETA_B}
if CARPETA_C is not None:
    versiones["05C"] = CARPETA_C
versiones = {k: v for k, v in versiones.items() if v is not None and Path(v).is_dir()}
for variante in variantes:
    print("\n" + variante)
    display(v04.comparar_versiones(versiones, variante))

In [ ]:
for etiqueta, carpeta in [("05A", CARPETA_A), ("05B", CARPETA_B)]:
    v05.registrar_evolucion(RAIZ, carpeta, "densa", etiqueta,
                            "Corpus ampliado" if etiqueta == "05A" else "Corpus ampliado y saneo por cita")
display(pd.DataFrame(json.loads((RAIZ / "configs/evolucion.json").read_text(encoding="utf-8"))))

## 10. Juez de texto libre

Solo sobre la mejor corrida en densa. Es la única medida de los 30 puntos de texto libre
y es lo que decide entre las dos políticas de generación. Cuesta llamadas de API, así que
va detrás de `usar_juez`.

In [ ]:
if usar_juez:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(RAIZ / "data/oficial/scripts/requirements-evaluador.txt")], check=True)
    mejores = {e: json.loads((c / "densa" / "resumen.json").read_text(encoding="utf-8"))["total_50"]
               for e, c in [("05A", CARPETA_A), ("05B", CARPETA_B)]}
    etiqueta = max(mejores, key=mejores.get)
    destino = (CARPETA_A if etiqueta == "05A" else CARPETA_B) / "densa"
    salida = destino / "evaluacion_con_ragas.json"
    if not salida.is_file():
        proceso = subprocess.run([sys.executable, str(RAIZ / "data/oficial/scripts/evaluate.py"),
                                  "--submission", str((destino / "submissions.jsonl").resolve()),
                                  "--split", "sample", "--ragas", "--out", str(salida.resolve())],
                                 cwd=RAIZ / "data/oficial", capture_output=True, text=True, encoding="utf-8")
        if proceso.returncode:
            raise RuntimeError(proceso.stderr[-2000:])
    print(etiqueta, json.loads(salida.read_text(encoding="utf-8"))["correccion_ragas"])
    display(v04.comparar_versiones(versiones, "densa"))

## Resultados

Pendientes de esta ejecución. Al terminar anotamos aquí, con cifras: cobertura del banco
antes y después, puntaje sin juez de 05A y 05B por variante, segundos por pregunta frente
a los 22 s y qué dijo el juez.

## Decisiones

- Si 05A sube frente al 04, el corpus ampliado entra a la entrega.
- Si el juez prefiere 05B, la política de saneo por cita reemplaza la del 04. Si no,
  mantenemos la del 04.
- Elegir variante de recuperación solo después, con todo lo demás congelado.
- Confirmar con el organizador si Qwen3-8B es admisible antes de considerar la corrida C.